In [4]:
import pandas as pd
import sqlite3
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('/content/telco_churn_cleaned.csv')
conn = sqlite3.connect(':memory:')
df.to_sql('customers', conn, index=False, if_exists='replace')

print("Database ready!")
print(f"Table 'customers' loaded with {len(df)} rows")

def query(sql):
    return pd.read_sql_query(sql, conn)

Database ready!
Table 'customers' loaded with 7043 rows


In [5]:
result = query("""
SELECT
    COUNT(*)                                        AS total_customers,
    SUM(Churn_Binary)                               AS churned,
    COUNT(*) - SUM(Churn_Binary)                    AS retained,
    ROUND(AVG(Churn_Binary) * 100, 2)               AS churn_rate_pct
FROM customers
""")
print(result.to_string(index=False))

 total_customers  churned  retained  churn_rate_pct
            7043     1869      5174           26.54


In [6]:
result = query("""
SELECT
    Contract,
    COUNT(*)                                        AS total_customers,
    SUM(Churn_Binary)                               AS churned,
    ROUND(AVG(Churn_Binary) * 100, 2)               AS churn_rate_pct
FROM customers
GROUP BY Contract
ORDER BY churn_rate_pct DESC
""")
print(result.to_string(index=False))

      Contract  total_customers  churned  churn_rate_pct
Month-to-month             3875     1655           42.71
      One year             1473      166           11.27
      Two year             1695       48            2.83


In [7]:
result = query("""
SELECT
    InternetService,
    COUNT(*)                                        AS total_customers,
    ROUND(AVG(Churn_Binary) * 100, 2)               AS churn_rate_pct,
    ROUND(AVG(MonthlyCharges), 2)                   AS avg_monthly_charge
FROM customers
GROUP BY InternetService
ORDER BY churn_rate_pct DESC
""")
print(result.to_string(index=False))

InternetService  total_customers  churn_rate_pct  avg_monthly_charge
    Fiber optic             3096           41.89               91.50
            DSL             2421           18.96               58.10
             No             1526            7.40               21.08


In [8]:
result = query("""
SELECT
    CASE
        WHEN tenure BETWEEN 0  AND 12 THEN '1. Early (0-12 months)'
        WHEN tenure BETWEEN 13 AND 24 THEN '2. Growing (13-24 months)'
        WHEN tenure BETWEEN 25 AND 48 THEN '3. Mature (25-48 months)'
        ELSE                                '4. Loyal (49+ months)'
    END                                             AS tenure_segment,
    COUNT(*)                                        AS total_customers,
    SUM(Churn_Binary)                               AS churned,
    ROUND(AVG(Churn_Binary) * 100, 2)               AS churn_rate_pct,
    ROUND(AVG(MonthlyCharges), 2)                   AS avg_monthly_charge
FROM customers
GROUP BY tenure_segment
ORDER BY tenure_segment
""")
print(result.to_string(index=False))

           tenure_segment  total_customers  churned  churn_rate_pct  avg_monthly_charge
   1. Early (0-12 months)             2186     1037           47.44               56.10
2. Growing (13-24 months)             1024      294           28.71               61.36
 3. Mature (25-48 months)             1594      325           20.39               65.93
    4. Loyal (49+ months)             2239      213            9.51               73.95


In [9]:
result = query("""
SELECT
    Contract,
    InternetService,
    PaymentMethod,
    COUNT(*)                                        AS high_value_churned,
    ROUND(AVG(MonthlyCharges), 2)                   AS avg_monthly_charge,
    ROUND(SUM(MonthlyCharges), 2)                   AS monthly_revenue_lost
FROM customers
WHERE Churn_Binary = 1
  AND MonthlyCharges > (SELECT AVG(MonthlyCharges) FROM customers)
GROUP BY Contract, InternetService, PaymentMethod
ORDER BY monthly_revenue_lost DESC
LIMIT 10
""")
print(result.to_string(index=False))

      Contract InternetService             PaymentMethod  high_value_churned  avg_monthly_charge  monthly_revenue_lost
Month-to-month     Fiber optic          Electronic check                 789               86.54              68281.50
Month-to-month     Fiber optic Bank transfer (automatic)                 149               87.67              13062.10
Month-to-month     Fiber optic   Credit card (automatic)                 122               87.96              10731.15
Month-to-month     Fiber optic              Mailed check                 102               82.42               8407.25
      One year     Fiber optic          Electronic check                  51              100.97               5149.65
      One year     Fiber optic   Credit card (automatic)                  23              104.02               2392.35
      One year     Fiber optic Bank transfer (automatic)                  23              101.93               2344.35
      Two year     Fiber optic Bank transfer (au

In [10]:
result = query("""
SELECT
    InternetService,
    Contract,
    COUNT(*)                                        AS customers,
    ROUND(AVG(Churn_Binary) * 100, 2)               AS churn_rate_pct,
    RANK() OVER (
        PARTITION BY InternetService
        ORDER BY AVG(Churn_Binary) DESC
    )                                               AS risk_rank
FROM customers
GROUP BY InternetService, Contract
ORDER BY InternetService, risk_rank
""")
print(result.to_string(index=False))

InternetService       Contract  customers  churn_rate_pct  risk_rank
            DSL Month-to-month       1223           32.22          1
            DSL       One year        570            9.30          2
            DSL       Two year        628            1.91          3
    Fiber optic Month-to-month       2128           54.61          1
    Fiber optic       One year        539           19.29          2
    Fiber optic       Two year        429            7.23          3
             No Month-to-month        524           18.89          1
             No       One year        364            2.47          2
             No       Two year        638            0.78          3


In [11]:
result = query("""
SELECT
    tenure,
    churned_this_month,
    SUM(churned_this_month) OVER (ORDER BY tenure)  AS cumulative_churned,
    ROUND(
        100.0 * SUM(churned_this_month) OVER (ORDER BY tenure)
        / SUM(churned_this_month) OVER (), 1
    )                                               AS pct_of_total_churn
FROM (
    SELECT tenure,
           SUM(Churn_Binary) AS churned_this_month
    FROM customers
    GROUP BY tenure
)
ORDER BY tenure
LIMIT 20
""")
print(result.to_string(index=False))

 tenure  churned_this_month  cumulative_churned  pct_of_total_churn
      0                   0                   0                 0.0
      1                 380                 380                20.3
      2                 123                 503                26.9
      3                  94                 597                31.9
      4                  83                 680                36.4
      5                  64                 744                39.8
      6                  40                 784                41.9
      7                  51                 835                44.7
      8                  42                 877                46.9
      9                  46                 923                49.4
     10                  45                 968                51.8
     11                  31                 999                53.5
     12                  38                1037                55.5
     13                  38                1075 

In [12]:
result = query("""
SELECT
    Contract,
    SeniorCitizen,
    COUNT(*)                                        AS total_customers,
    SUM(Churn_Binary)                               AS churned,
    ROUND(AVG(Churn_Binary) * 100, 2)               AS churn_rate_pct,
    ROUND(SUM(CASE WHEN Churn_Binary=1
                   THEN MonthlyCharges ELSE 0 END),
          2)                                        AS monthly_revenue_lost,
    ROUND(SUM(CASE WHEN Churn_Binary=1
                   THEN MonthlyCharges * 12 ELSE 0 END),
          2)                                        AS annualized_revenue_lost
FROM customers
GROUP BY Contract, SeniorCitizen
ORDER BY monthly_revenue_lost DESC
LIMIT 8
""")
print(result.to_string(index=False))

      Contract SeniorCitizen  total_customers  churned  churn_rate_pct  monthly_revenue_lost  annualized_revenue_lost
Month-to-month            No             3068     1214           39.57              85735.85                1028830.2
Month-to-month           Yes              807      441           54.65              35111.25                 421335.0
      One year            No             1283      137           10.68              11345.25                 136143.0
      Two year            No             1550       42            2.71               3630.15                  43561.8
      One year           Yes              190       29           15.26               2773.20                  33278.4
      Two year           Yes              145        6            4.14                535.15                   6421.8


In [13]:
result = query("""
SELECT
    Churn                                           AS status,
    COUNT(*)                                        AS customers,
    ROUND(AVG(tenure), 1)                           AS avg_tenure_months,
    ROUND(AVG(MonthlyCharges), 2)                   AS avg_monthly_charge,
    ROUND(AVG(TotalCharges), 2)                     AS avg_total_revenue,
    ROUND(100.0 * SUM(CASE WHEN Contract='Month-to-month'
                           THEN 1 ELSE 0 END) / COUNT(*), 1)
                                                    AS pct_month_to_month,
    ROUND(100.0 * SUM(CASE WHEN SeniorCitizen='Yes'
                           THEN 1 ELSE 0 END) / COUNT(*), 1)
                                                    AS pct_senior
FROM customers
GROUP BY Churn
""")
print(result.to_string(index=False))

status  customers  avg_tenure_months  avg_monthly_charge  avg_total_revenue  pct_month_to_month  pct_senior
    No       5174               37.6               61.27            2549.91                42.9        12.9
   Yes       1869               18.0               74.44            1531.80                88.6        25.5


In [14]:
import os
os.makedirs('sql', exist_ok=True)

queries = {
    'churn_by_contract': """
        SELECT Contract, COUNT(*) total, SUM(Churn_Binary) churned,
               ROUND(AVG(Churn_Binary)*100,2) churn_rate_pct,
               ROUND(AVG(MonthlyCharges),2) avg_charge
        FROM customers GROUP BY Contract ORDER BY churn_rate_pct DESC""",
    'churn_by_tenure_segment': """
        SELECT CASE
            WHEN tenure BETWEEN 0  AND 12 THEN 'Early (0-12m)'
            WHEN tenure BETWEEN 13 AND 24 THEN 'Growing (13-24m)'
            WHEN tenure BETWEEN 25 AND 48 THEN 'Mature (25-48m)'
            ELSE 'Loyal (49+m)' END AS segment,
            COUNT(*) total, SUM(Churn_Binary) churned,
            ROUND(AVG(Churn_Binary)*100,2) churn_rate_pct
        FROM customers GROUP BY segment ORDER BY segment""",
    'revenue_at_risk': """
        SELECT Contract, InternetService,
               ROUND(SUM(CASE WHEN Churn_Binary=1
                   THEN MonthlyCharges ELSE 0 END),2) monthly_revenue_lost
        FROM customers GROUP BY Contract, InternetService
        ORDER BY monthly_revenue_lost DESC""",
    'customer_profile_comparison': """
        SELECT Churn, COUNT(*) customers,
               ROUND(AVG(tenure),1) avg_tenure,
               ROUND(AVG(MonthlyCharges),2) avg_monthly_charge,
               ROUND(AVG(TotalCharges),2) avg_total_revenue
        FROM customers GROUP BY Churn"""
}

for name, sql in queries.items():
    pd.read_sql_query(sql, conn).to_csv(f'sql/{name}.csv', index=False)
    print(f"Saved: sql/{name}.csv")

conn.close()
print("\nAll SQL exports done. Database connection closed.")

Saved: sql/churn_by_contract.csv
Saved: sql/churn_by_tenure_segment.csv
Saved: sql/revenue_at_risk.csv
Saved: sql/customer_profile_comparison.csv

All SQL exports done. Database connection closed.
